In [ ]:
import sys
from pathlib import Path

# Add parent directory to path to enable imports
parent_dir = Path().absolute().parent
if str(parent_dir) not in sys.path:
	sys.path.insert(0, str(parent_dir))

from data_loader import create_training_dataloader
from model import *
from tqdm import tqdm
import torch
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

In [ ]:
class GWOpticalContrastiveModel(nn.Module):
    """
    End-to-End Model Wrapper for Contrastive Training.
    Combines:
      1. GW Encoder (Scalar MLP + Skymap ResNet)
      2. Optical Encoder (mTAN + Attention)
      3. Alignment Head (Projection + Loss)
    """
    def __init__(self, 
                 gw_scalar_dim=7, 
                 gw_skymap_channels=7, 
                 optical_input_dim=6, 
                 ref_time_dim=64,
                 enc_dim=128, 
                 proj_dim=256,
                 temp_init=0.07):
        super().__init__()
        
        # --- 1. Encoders ---
        self.gw_encoder = GWMOCResNetEncoder(
            scalar_input_dim=gw_scalar_dim,
            skymap_channels=gw_skymap_channels,
            final_output_dim=enc_dim
        )
        
        self.optical_encoder = OpticalEncoderWithCLS(
            input_dim=optical_input_dim,
            output_dim=enc_dim,
            num_heads=4,
            ref_dim=ref_time_dim
        )
        
        # --- 2. Projection Heads & Temperature ---
        # Projects Encoder Features (128) -> Latent Space (256)
        self.gw_proj = ProjectionHead(enc_dim, enc_dim, proj_dim)
        self.opt_proj = ProjectionHead(enc_dim, enc_dim, proj_dim)
        
        # We store log_temp to ensure temperature is always positive (via exp)
        self.log_temp = nn.Parameter(torch.ones([]) * torch.log(torch.tensor(temp_init)))

        # --- 3. Standard Cross Entropy Loss ---
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, gw_s, gw_m, opt_t, opt_v, opt_ref_t, opt_mask, opt_err, opt_coords, gw_indices, mask=None):
        """
        Args:
            gw_s, gw_m: GW Inputs
            opt_t, opt_v, ...: Optical Inputs
            gw_indices: [Batch] ID of the GW event (for masking)
        """
        # --- A. Encode Features ---
        # g: [Batch, enc_dim]
        g = self.gw_encoder(gw_s, gw_m)
        
        # z_l: [Batch, enc_dim] (We discard H_l for contrastive pre-training)
        z_l, _ = self.optical_encoder(opt_coords, opt_t, opt_v, opt_ref_t, opt_mask, errors_obs=opt_err)
        
        # --- B. Project & Normalize ---
        # [Batch, proj_dim]
        feat_g = F.normalize(self.gw_proj(g), dim=1)
        feat_o = F.normalize(self.opt_proj(z_l), dim=1)
        
        # --- C. Compute Contrastive Loss ---
        loss, logits = self.compute_masked_itc_loss(feat_g, feat_o, gw_indices, mask=None)
        
        return loss, logits

    def compute_masked_itc_loss(self, feat_g, feat_o, gw_indices, mask=None):
        """
        Computes Image-Text Contrastive (ITC) loss.
        Uses gw_indices to handle potential 'false negatives' 
        (though BalancedSampler avoids them, this is robust).
        """
        batch_size = feat_g.size(0)
        logit_scale = torch.clamp(self.log_temp.exp(), min=0.01, max=100.0)
        
        # 1. Similarity Matrix: [B, B]
        sim_g2o = torch.matmul(feat_g, feat_o.T) * logit_scale
        sim_o2g = sim_g2o.T
        
        if mask is not None:
            # 2. Ground Truth Mask: [B, B]
            # mask[i, j] = 1 if sample i and j come from the same GW event
            # If using BalancedSampler, this is just an identity matrix.
            labels_mask = (gw_indices.unsqueeze(0) == gw_indices.unsqueeze(1)).float()
            
            # 3. Compute Loss (Masked Cross Entropy)
            # We want to maximize similarity for all positive pairs (where mask == 1)
            
            # For numerical stability with Softmax
            sim_g2o_max, _ = torch.max(sim_g2o, dim=1, keepdim=True)
            sim_g2o = sim_g2o - sim_g2o_max.detach()
            
            sim_o2g_max, _ = torch.max(sim_o2g, dim=1, keepdim=True)
            sim_o2g = sim_o2g - sim_o2g_max.detach()
            
            # Log-Softmax denominator (sum over all samples in batch)
            exp_g2o = torch.exp(sim_g2o)
            exp_o2g = torch.exp(sim_o2g)
            
            # Note: If BalancedSampler is used, this simplifies to standard CE.
            # Here we implement the generic form for safety.
            
            # Log-prob of positive pairs
            # sum(exp(positives)) / sum(exp(all))
            log_prob_g2o = sim_g2o - torch.log(exp_g2o.sum(dim=1, keepdim=True))
            log_prob_o2g = sim_o2g - torch.log(exp_o2g.sum(dim=1, keepdim=True))
            
            # Compute mean loss over positive pairs
            # We only care about entries where labels_mask == 1
            loss_g = - (labels_mask * log_prob_g2o).sum(dim=1) / labels_mask.sum(dim=1)
            loss_o = - (labels_mask * log_prob_o2g).sum(dim=1) / labels_mask.sum(dim=1)

            total_loss = (loss_g.mean() + loss_o.mean()) / 2
        else:
            # 4. Standard Cross Entropy Labels
            labels = torch.arange(batch_size, device=feat_g.device)
            # Compute Symmetric Loss
            # Loss 1: Given GW, classify correct Optical
            loss_g = self.criterion(sim_g2o, labels)
            # Loss 2: Given Optical, classify correct GW
            loss_o = self.criterion(sim_o2g, labels)
            
            total_loss = (loss_g + loss_o) / 2
        
        return total_loss, sim_g2o

In [ ]:
h5file = "<BASE_DIR>/data/LSST_KN_BNS/combined_dataset.h5"
dataloader = create_training_dataloader(
    h5_path=h5file,
    batch_size=32,
    steps_per_epoch=10000
)
model = GWOpticalContrastiveModel(
    gw_scalar_dim=7,
    gw_skymap_channels=7, # Assuming robust preprocessing output
    optical_input_dim=6,
    enc_dim=128,
    proj_dim=256
)
print("\nTesting Model Forward Pass...")
for batch_idx, batch_data in enumerate(dataloader):
    # Unpack data
    # Order matches __getitem__: scalar, skymap, time, val, mask, err, gw_idx
    gw_s, gw_m, opt_t, opt_v, opt_mask, opt_err, opt_coords, gw_indices = batch_data
    opt_ref_t = torch.linspace(-0.3, 0.6, 64).unsqueeze(0).repeat(gw_s.size(0), 1)
    
    # Forward Pass
    loss, logits = model(
        gw_s, gw_m,
        opt_t, opt_v,
        opt_ref_t, opt_mask,
        opt_err, opt_coords,
        gw_indices
    )
    
    print(f"Batch {batch_idx}:")
    print(f"  Loss: {loss.item()}")
    print(f"  Logits Shape: {logits.shape}") # Expected: [B, B]

    loss.backward()
    
    break

In [ ]:
data_path = '<BASE_DIR>/data/LSST_KN_BNS/combined_dataset.h5'
batch_size = 32
num_workers = 4
epochs = 10
# --- 1. Setup Device & Config ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

In [ ]:
# --- 2. Data Preparation ---
# Calculate correct steps_per_epoch based on optical data volume
import h5py
with h5py.File(data_path, 'r') as f:
    total_optical = f['events/optical_data/values'].shape[0]

steps_per_epoch = total_optical // batch_size
print(f"Dataset Size: {total_optical} | Steps/Epoch: {steps_per_epoch}")

train_loader = create_training_dataloader(
    h5_path=data_path,
    batch_size=batch_size,
    steps_per_epoch=10,
    num_workers=num_workers
)

In [ ]:
# --- 3. Model Initialization ---
model = GWOpticalContrastiveModel(
    gw_scalar_dim=7,
    gw_skymap_channels=7, # Assuming robust preprocessing output
    optical_input_dim=6,
    enc_dim=128,
    proj_dim=256
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Mixed Precision Scaler
scaler = torch.amp.GradScaler(device=device.type)

# --- 4. Training Loop ---
model.train()

for epoch in range(2):
    epoch_loss = 0.0
    
    # Tqdm progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for batch_idx, batch_data in enumerate(pbar):
        # Unpack data
        gw_s, gw_m, opt_t, opt_v, opt_mask, opt_err, opt_coords, gw_indices = [x.to(device) for x in batch_data]
        
        # Create Reference Time Query (Learned queries need t_ref)
        # For mTAN, we typically query at the same observed times OR fixed grid.
        # Here we query at observed times for reconstruction/encoding.
        # (Note: In pure contrastive learning, we just encode the sequence)
        # Assuming Encoder implementation uses fixed reference points internally 
        # or we pass t_obs as t_ref to get representations at specific points.
        # BUT: OpticalEncoderWithCLS usually expects t_ref to generate the queries.
        # Simple strategy: Use linspace 0-1 as reference time (normalized)
        B = gw_s.size(0)
        N_ref = 64 # Number of reference points
        opt_ref_t = torch.linspace(-0.3, 0.6, N_ref).unsqueeze(0).repeat(B, 1).to(device)
        
        optimizer.zero_grad()
        
        # Mixed Precision Forward
        with torch.amp.autocast(device_type=device.type):
            loss, logits = model(
                gw_s, gw_m, 
                opt_t, opt_v, opt_ref_t, 
                opt_mask, opt_err, opt_coords,
                gw_indices
            )
        
        # Backward
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Logging
        epoch_loss += loss.item()
        
        if batch_idx % 10 == 0:
            # Calculate simple accuracy (are diagonal elements the largest?)
            # Only valid if BalancedSampler ensures unique GWs
            preds = torch.argmax(logits, dim=1)
            targets = torch.arange(B).to(device)
            acc = (preds == targets).float().mean()
            
            pbar.set_postfix({
                'Loss': f"{loss.item():.4f}", 
                'Acc': f"{acc.item():.2f}",
                'Temp': f"{model.log_temp.exp().item():.2f}"
            })
    
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} Complete. Avg Loss: {avg_loss:.4f}")
    
    # Save Checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
    }, f"checkpoint_epoch_{epoch+1}.pth")